# Final submission

Trains the search-selected best model on the full training set and writes a submission file: CatBoost binary classifier, with both a manual fold-safe per-`ALLOCATION` target encoding *and* `ALLOCATION`/`GROUP` passed as CatBoost's own native categorical features (ordered target statistics, computed leak-safely inside `.fit()` from training rows only -- a different algorithm than the manual encoding, tried once the manual-encoding-only ceiling proved robust across seven other independent angles). ~0.526 `TS`-grouped CV accuracy either way; this combined version is adopted because it's the most fold-stable (tightest spread across the 5 folds) of everything tried, not because its mean beats the rest -- see `notes/accuracy_ceiling.md`.

## Setup

In [1]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import catboost as cb

import qrt_prep as P
import qrt_features as F

## Load, engineer, train

The allocation encoding is computed once from the *entire* training set (legitimate here, unlike inside CV, because every test-set row's `ALLOCATION` already appears in train -- 100% overlap -- and no test `TARGET` is used). `ALLOCATION`/`GROUP` are additionally passed as native categoricals so CatBoost's own ordered target statistics can pick up anything the manual encoding misses.

In [2]:
X_train, y_train, X_test = P.load_raw('../data/raw')
train_df = F.engineer(X_train.join(y_train))
test_df = F.engineer(X_test)

NUM_FEATURES = (P.BASE_FEATURES + F.GROUP_DUMMY_COLS + F.MISSING_COLS
                + F.ROLLING_COLS + F.CROSS_SECTIONAL_COLS)
train_enc, test_enc = F.add_alloc_encoding(train_df, test_df, k=50)
train_df['ALLOC_ENC'] = train_enc
test_df['ALLOC_ENC'] = test_enc
train_df['ALLOCATION_CAT'] = train_df['ALLOCATION'].astype(str)
test_df['ALLOCATION_CAT'] = test_df['ALLOCATION'].astype(str)
train_df['GROUP_CAT'] = train_df['GROUP'].astype(str)
test_df['GROUP_CAT'] = test_df['GROUP'].astype(str)
for c in NUM_FEATURES:
    train_df[c] = train_df[c].fillna(0)
    test_df[c] = test_df[c].fillna(0)

FEATURES = NUM_FEATURES + ['ALLOC_ENC', 'ALLOCATION_CAT', 'GROUP_CAT']
CAT_FEATURES = ['ALLOCATION_CAT', 'GROUP_CAT']

y_sign = (train_df['target'] > 0).astype(int).to_numpy()
len(FEATURES)

54

In [3]:
model = cb.CatBoostClassifier(iterations=300, learning_rate=0.015, depth=6,
                               random_seed=42, verbose=False, thread_count=8,
                               cat_features=CAT_FEATURES)
model.fit(train_df[FEATURES], y_sign)

CatBoostClassifier(cat_features=['ALLOCATION_CAT', 'GROUP_CAT'], depth=6, iterations=300, learning_rate=0.015, random_seed=42, thread_count=8, verbose=False)

## Predict and format

In [4]:
proba = model.predict_proba(test_df[FEATURES])[:, 1]
submission = pd.DataFrame({'prediction': (proba > 0.5).astype(int)}, index=test_df.index)
submission.index.name = 'ROW_ID'
submission['prediction'].value_counts(normalize=True)

prediction
1    0.583903
0    0.416097
Name: proportion, dtype: float64

## Sanity checks against sample_submission.csv

In [5]:
sample_submission = pd.read_csv('../data/raw/sample_submission.csv', index_col='ROW_ID')

assert submission.shape == sample_submission.shape
assert (submission.index == sample_submission.index).all()
assert set(submission['prediction'].unique()) <= {0, 1}
print('shape ok, index aligned, values in {0, 1}')
print('predicted positive share:', submission['prediction'].mean())
print('sample_submission positive share (random):', sample_submission['prediction'].mean())

shape ok, index aligned, values in {0, 1}
predicted positive share: 0.5839033573893944
sample_submission positive share (random): 0.5014747411358644


## Write

In [6]:
submission.to_csv('../submissions/catboost_alloc_encoding.csv')
pd.read_csv('../submissions/catboost_alloc_encoding.csv').head()

,ROW_ID,prediction
0,527073,0
1,527074,1
2,527075,0
3,527076,0
4,527077,0


## CV accuracy of this exact pipeline

Re-verifies the search's headline number (`notes/accuracy_ceiling.md`) using this notebook's own code, on the standard 5-fold `TS`-grouped CV.

In [7]:
folds = P.make_folds(train_df['TS'], n_splits=5, seed=0)
y = train_df['target'].to_numpy()
accs = []
for fold in range(5):
    val_mask = folds == fold
    tr, va = train_df[~val_mask], train_df[val_mask]
    tr_enc, va_enc = F.add_alloc_encoding(tr, va, k=50)
    tr = tr.copy(); va = va.copy()
    tr['ALLOC_ENC'] = tr_enc
    va['ALLOC_ENC'] = va_enc
    fold_model = cb.CatBoostClassifier(iterations=300, learning_rate=0.015, depth=6,
                                        random_seed=42, verbose=False, thread_count=8,
                                        cat_features=CAT_FEATURES)
    fold_model.fit(tr[FEATURES], (y[~val_mask] > 0).astype(int))
    pred = fold_model.predict_proba(va[FEATURES])[:, 1]
    accs.append(((pred > 0.5).astype(int) == (y[val_mask] > 0).astype(int)).mean())

print('mean CV accuracy:', np.mean(accs), accs)

mean CV accuracy: 0.5257805664238274 [np.float64(0.5260796153210857), np.float64(0.5246181613523008), np.float64(0.5274379521107214), np.float64(0.5256169134950713), np.float64(0.5251501898399578)]


This CatBoost pipeline (manual allocation encoding + native categoricals + cross-sectional features) reaches ~0.526 mean CV accuracy with the tightest fold-to-fold spread of any config tried -- the best validated result from an extensive search (150+ configurations across feature engineering, 3 GBM families, hierarchical encoding, ensembling, neural sequence models, and CatBoost's native categorical handling; see `notes/accuracy_ceiling.md`). It comfortably beats the published benchmark's 0.5079 public score, but the search found this to be a real ceiling for this feature set, not a starting point for much further improvement.